#### In this notebook, we'll load and take an initial look into the data

In [4]:
# Importing packages and loading data
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# connect to the database
conn = sqlite3.connect("../data/aqi.db")

# load cleaned data
df = pd.read_sql("""
    SELECT city, timestamp, value, is_imputed, is_outlier_flag
    FROM cleaned_readings
    ORDER BY city, timestamp
""", conn)

conn.close()

# convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(f"total rows: {len(df)}")
print(f"date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"cities: {df['city'].unique()}")
print(f"\n{df.groupby('city')['value'].describe().round(2)}")

total rows: 864
date range: 2026-06-20 00:00:00 to 2026-07-01 23:00:00
cities: <StringArray>
['Chicago', 'Los Angeles', 'San Jose']
Length: 3, dtype: str

             count   mean   std  min    25%    50%    75%   max
city                                                           
Chicago      288.0  12.68  7.00  2.2   8.00  10.80  17.23  47.4
Los Angeles  288.0  13.94  4.13  4.3  11.18  13.95  16.80  25.3
San Jose     288.0   8.65  3.08  1.5   6.48   9.10  10.82  14.8


In [5]:
# check what's actually in raw_readings
conn = sqlite3.connect("../data/aqi.db")

raw_check = pd.read_sql("""
    SELECT city, 
           COUNT(*) as rows,
           MIN(timestamp) as earliest,
           MAX(timestamp) as latest
    FROM raw_readings
    GROUP BY city
""", conn)

conn.close()
print(raw_check)

          city  rows                    earliest                      latest
0      Chicago   288  2026-06-20 00:00:00.000000  2026-07-01 23:00:00.000000
1  Los Angeles   288  2026-06-20 00:00:00.000000  2026-07-01 23:00:00.000000
2     San Jose   288  2026-06-20 00:00:00.000000  2026-07-01 23:00:00.000000


In [6]:
import requests

# test how far back Open-Meteo actually gives us data
resp = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude":  37.34,
        "longitude": -121.89,
        "hourly":    "pm2_5",
        "timezone":  "America/Los_Angeles",
        "past_days": 90,
    }
)

data = resp.json()
times = data["hourly"]["time"]
print(f"first timestamp: {times[0]}")
print(f"last timestamp:  {times[-1]}")
print(f"total hours returned: {len(times)}")

first timestamp: 2026-03-30T00:00
last timestamp:  2026-07-02T23:00
total hours returned: 2280
